In [1]:
import numpy as np
import pandas as pd

import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

import kagglehub


/kaggle/input/datasets/yannahomlanto/esgc-vak-sujet-07/Sujet_07_Santander_Transaction-20261110T150120Z-1-001/Sujet_07_Santander_Transaction/Carnet_de_demarrage.ipynb
/kaggle/input/datasets/yannahomlanto/esgc-vak-sujet-07/Sujet_07_Santander_Transaction-20261110T150120Z-1-001/Sujet_07_Santander_Transaction/Enonce.md
/kaggle/input/datasets/yannahomlanto/esgc-vak-sujet-07/Sujet_07_Santander_Transaction-20261110T150120Z-1-001/Sujet_07_Santander_Transaction/a_decider.csv
/kaggle/input/datasets/yannahomlanto/esgc-vak-sujet-07/Sujet_07_Santander_Transaction-20261110T150120Z-1-001/Sujet_07_Santander_Transaction/train.csv


In [2]:
import pandas as pd

DOSSIER = "/kaggle/input/datasets/yannahomlanto/esgc-vak-sujet-07/Sujet_07_Santander_Transaction-20261110T150120Z-1-001/Sujet_07_Santander_Transaction/"

train = pd.read_csv(DOSSIER + "train.csv")
a_decider = pd.read_csv(DOSSIER + "a_decider.csv")

print("train     :", train.shape)
print("a_decider :", a_decider.shape)

train     : (160000, 202)
a_decider : (40000, 201)


> **Usage de l'IA.** Les outils de suivi, les mesures de 1.1 et 1.8, et les étapes 2.1, 2.4 et 2.7 ont été écrites avec l'aide de Claude (Anthropic), puis relues, exécutées et vérifiées par l'équipe.

## 0 — Outils de suivi

Copie conforme de `notebook_Outils_de_suivi.ipynb`. Chaque outil remplit un tableau qu'on rend :

| Outil | La question |
|---|---|
| `prevoir` / `noter` / `essai` | qu'ai-je annoncé, et l'ai-je mesuré après ? |
| `constat` | qu'ai-je vu dans les données, avec quel chiffre ? |
| `hypothese` | qu'est-ce que je crois, et qu'est-ce qui me contredirait ? |
| `journal` | qu'ai-je décidé, et pourquoi, sans parler du score ? |

Seul changement : `barre` (la barre de bruit) sera mesurée à l'étape 2.2. En attendant elle vaut `None`, et `journal` l'indique au lieu de planter.

In [3]:
# 0 — outils de suivi (copie conforme de notebook_Outils_de_suivi)
from datetime import datetime
from IPython.display import display

ID = "ID_code"     # la colonne identifiant
CIBLE = "target"   # la colonne à prédire

# la barre de bruit sera mesurée à l'étape 2.2 : None en attendant
barre = None

ESSAIS = []
PREDICTIONS = []

def prevoir(essai, attendu, raison):
    """Écrit ce que vous attendez AVANT de mesurer — horodaté."""
    PREDICTIONS[:] = [p for p in PREDICTIONS if p['essai'] != essai]
    PREDICTIONS.append({'essai': essai, 'attendu': attendu, 'raison': raison,
                        'écrit à': datetime.now().strftime('%H:%M:%S')})
    display(pd.DataFrame(PREDICTIONS))

def noter(hypothese, changement, attendu, score):
    """Ajoute une ligne au tableau. Refuse un essai non annoncé."""
    prevu = [p for p in PREDICTIONS if p['essai'] == hypothese]
    if not prevu:
        print(f'  refusé : aucune prédiction pour « {hypothese} ».')
        print('  Écrivez prevoir(...) d\'abord — l\'ordre est vérifié, pas seulement demandé.')
        return
    ligne = {'hypothèse': hypothese, 'changement': changement, 'attendu': attendu,
             'mesuré': round(score, 2),
             'prédiction écrite avant ?': 'oui, à ' + prevu[0]['écrit à']}
    ESSAIS[:] = [e for e in ESSAIS if e['hypothèse'] != hypothese] + [ligne]
    display(pd.DataFrame(ESSAIS))

def essai(nom, mesure, changement='', attendu=''):
    """Le chemin strict : la mesure est prise ici, donc l'ordre est prouvé."""
    prevu = [p for p in PREDICTIONS if p['essai'] == nom]
    if not prevu:
        print(f'refusé : aucune prédiction pour « {nom} ». Écrivez prevoir(...) d\'abord.')
        return None
    score = 100 * mesure()
    noter(nom, changement or 'voir la question', attendu or prevu[0]['attendu'], score)
    return score

CONSTATS = []

def constat(observation, chiffre, comparaison, consequence):
    """observation : ce qu'on a vu · chiffre : la mesure exacte
       comparaison : par rapport à quoi ? · conséquence : garder, écarter, surveiller."""
    ligne = {'observation': observation, 'chiffre': chiffre,
             'comparaison': comparaison, 'conséquence': consequence}
    CONSTATS[:] = [c for c in CONSTATS if c['observation'] != observation] + [ligne]
    display(pd.DataFrame(CONSTATS))

HYPOTHESES = []

def hypothese(feature, ce_qu_elle_raconte, comment_on_la_test, ce_qui_la_contredirait):
    """Une hypothèse est une prédiction qu'on peut tester, avec un critère annoncé AVANT."""
    ligne = {'feature': feature, 'ce qu\'elle raconte': ce_qu_elle_raconte,
             'test': comment_on_la_test, 'contredite si': ce_qui_la_contredirait}
    HYPOTHESES[:] = [h for h in HYPOTHESES if h['feature'] != feature] + [ligne]
    display(pd.DataFrame(HYPOTHESES))

JOURNAL = []

def journal(decision, choix, raison, effet):
    """Décision, choix, raison d'audit, effet mesuré — comparé au double de la barre."""
    JOURNAL.append({'décision': decision, 'choix': choix,
                    'raison (sans le score)': raison,
                    'effet mesuré': round(effet, 2),
                    # SEUL CHANGEMENT : tant que la barre n'est pas mesurée (2.2), on ne compare pas
                    '> double de la barre ?': ('oui' if abs(effet) > 2 * barre else 'non')
                                              if barre is not None else 'barre pas encore mesurée'})
    display(pd.DataFrame(JOURNAL))

## Étape 1.1 — Inventaire

On a reçu 4 fichiers, dont 2 tables de données. Les 2 autres sont l'énoncé (Enonce.md) et le carnet de démarrage (Carnet_de_demarrage.ipynb).

| Table | Lignes | Colonnes |
|---|---|---|
| train.csv | 160 000 | 202 |
| a_decider.csv | 40 000 | 201 |

a_decider a une colonne de moins : il lui manque `target`. Elle contient les clients sur lesquels on doit prédire ; si on avait leur target, il n'y aurait rien à prédire.

Les chiffres correspondent à ceux de l'énoncé, donc rien n'est tronqué.

In [4]:
print(set(train.columns) - set(a_decider.columns))

X = train.drop(columns=["ID_code", "target"])
print(X.shape)

{'target'}
(160000, 200)


In [5]:
# le constat de l'étape 1.1 : les dimensions comparées à celles de l'énoncé
conforme = train.shape == (160_000, 202) and a_decider.shape == (40_000, 201)
constat("les tables ont la taille annoncée",
        f"train {train.shape[0]} × {train.shape[1]}, a_decider {a_decider.shape[0]} × {a_decider.shape[1]}",
        "énoncé : 160 000 × 202 et 40 000 × 201",
        "conforme : rien n'est tronqué" if conforme else "ÉCART : retélécharger les données")

,observation,chiffre,comparaison,conséquence
0,les tables ont la taille annoncée,"train 160000 × 202, a_decider 40000 × 201",énoncé : 160 000 × 202 et 40 000 × 201,conforme : rien n'est tronqué


Le `constat` range ce résultat dans le tableau des constats : l'observation, le chiffre, la comparaison avec l'énoncé et la conséquence.

**À savoir dire sans carnet.** On compare les dimensions lues à celles de l'énoncé, parce qu'un fichier tronqué au téléchargement donne des scores faux sans aucune erreur. Si on oubliait la colonne `target` dans la comparaison (201 au lieu de 202 pour `train`), le constat crierait à tort.

## Étape 1.8 — Informations indisponibles au moment de la décision

Question posée à chaque colonne : le jour où la banque décide, cette valeur existe-t-elle déjà ?

| Colonne | Décision | Raison |
|---|---|---|
| `target` | mise à part (c'est la cible `y`) | c'est ce qu'on veut prédire : la banque ne la connaît pas au moment de décider. La laisser dans les colonnes du modèle, ce serait lui donner la réponse. |
| `ID_code` | écartée | c'est un identifiant : il désigne le client, il ne le décrit pas. Le modèle apprendrait des numéros par cœur, inutiles sur de nouveaux clients. |
| `var_0` … `var_199` | gardées | elles sont toutes présentes dans `a_decider.csv`, la table des clients sur qui on décide : elles existent donc au moment de la décision. |

Vérification : la seule colonne de `train` absente de `a_decider` est `target`. Après retrait, la table du modèle `X` fait 160 000 lignes et 200 colonnes.

Aucune valeur ne manque (voir étape 1.6) : le signe de fuite « colonne vide quand la réponse n'est pas connue » ne peut pas apparaître ici.

### Étape 1.8 (suite) — La mesure confirme

Les variables sont anonymes : on ne peut pas raisonner par le métier. Alors on **mesure** les deux signes de fuite vus en J2 §5, plus l'identifiant.

- **le vide** : une colonne remplie seulement quand l'issue est connue ;
- **la valeur** : une variable qui, seule, sépare presque parfaitement les deux classes (AUC > 0,90) ;
- **l'identifiant** : unique, et sans lien avec la cible.

In [6]:
from sklearn.metrics import roc_auc_score

y = train[CIBLE]

# signe du vide : combien de trous dans les deux tables ?
trous = int(train.isna().sum().sum() + a_decider.isna().sum().sum())
constat("signe du vide : aucune colonne n'a de trou", f"{trous} trou",
        f"sur {train.size + a_decider.size} cellules (train + a_decider)",
        "aucune colonne ne peut trahir la cible par son vide")

# signe de la valeur : l'AUC de chaque variable prise seule
auc = pd.Series({col: roc_auc_score(y, X[col]) for col in X.columns})
auc = np.maximum(auc, 1 - auc)   # une variable qui classe « à l'envers » sépare aussi
display(auc.sort_values(ascending=False).head(5).round(4).to_frame("AUC seule"))
constat("signe de la valeur : aucune variable seule ne connaît la réponse",
        f"AUC seule maximale {auc.max():.4f} ({auc.idxmax()})",
        f"seuil d'alarme 0,90 : {(auc > 0.90).sum()} variable au-dessus",
        "les 200 variables sont gardées")

# l'identifiant : unique ? son numéro dit-il quelque chose de la cible ?
numero = train[ID].str.extract(r"(\d+)$")[0].astype(int)   # "train_123" -> 123
auc_numero = roc_auc_score(y, numero)
communs = len(set(train[ID]) & set(a_decider[ID]))
constat("l'identifiant désigne une ligne, il ne décrit pas le client",
        f"{train[ID].nunique()} valeurs pour {len(train)} lignes ; AUC du numéro {max(auc_numero, 1 - auc_numero):.4f}",
        f"0,5 = hasard ; {communs} ID en commun avec a_decider",
        "l'écarter")

# la liste des colonnes écartées, avec leur raison
journal(ID, "écarté", "il désigne le client au lieu de le décrire : il ne se reproduit pas chez un nouveau client", 0.00)

,observation,chiffre,comparaison,conséquence
0,les tables ont la taille annoncée,"train 160000 × 202, a_decider 40000 × 201",énoncé : 160 000 × 202 et 40 000 × 201,conforme : rien n'est tronqué
1,signe du vide : aucune colonne n'a de trou,0 trou,sur 40360000 cellules (train + a_decider),aucune colonne ne peut trahir la cible par son...


,AUC seule
var_81,0.5681
var_139,0.5668
var_6,0.5597
var_12,0.5569
var_110,0.5569


,observation,chiffre,comparaison,conséquence
0,les tables ont la taille annoncée,"train 160000 × 202, a_decider 40000 × 201",énoncé : 160 000 × 202 et 40 000 × 201,conforme : rien n'est tronqué
1,signe du vide : aucune colonne n'a de trou,0 trou,sur 40360000 cellules (train + a_decider),aucune colonne ne peut trahir la cible par son...
2,signe de la valeur : aucune variable seule ne ...,AUC seule maximale 0.5681 (var_81),"seuil d'alarme 0,90 : 0 variable au-dessus",les 200 variables sont gardées


,observation,chiffre,comparaison,conséquence
0,les tables ont la taille annoncée,"train 160000 × 202, a_decider 40000 × 201",énoncé : 160 000 × 202 et 40 000 × 201,conforme : rien n'est tronqué
1,signe du vide : aucune colonne n'a de trou,0 trou,sur 40360000 cellules (train + a_decider),aucune colonne ne peut trahir la cible par son...
2,signe de la valeur : aucune variable seule ne ...,AUC seule maximale 0.5681 (var_81),"seuil d'alarme 0,90 : 0 variable au-dessus",les 200 variables sont gardées
3,"l'identifiant désigne une ligne, il ne décrit ...",160000 valeurs pour 160000 lignes ; AUC du num...,"0,5 = hasard ; 0 ID en commun avec a_decider",l'écarter


,décision,choix,raison (sans le score),effet mesuré,> double de la barre ?
0,ID_code,écarté,il désigne le client au lieu de le décrire : i...,0.0,barre pas encore mesurée


| Ce qu'on vérifie | Mesure | Conclusion |
|---|---|---|
| Signe du vide | 0 trou dans `train`, 0 dans `a_decider` | aucune colonne ne peut trahir la cible par son vide |
| Signe de la valeur | meilleure variable seule : `var_81`, AUC 0,5681 ; 0 variable au-dessus de 0,90 | aucune variable ne « connaît » la réponse : les 200 sont gardées |
| `ID_code` unique ? | 160 000 valeurs pour 160 000 lignes | il désigne une ligne, il ne la décrit pas |
| `ID_code` lié à la cible ? | AUC de son numéro : 0,502 | pur hasard : le numéro ne dit rien du client |
| Client présent des deux côtés ? | 0 `ID_code` en commun avec `a_decider` | aucun client n'est à la fois appris et à décider |

**Colonnes écartées : `ID_code` seul.** Ce n'est pas une supposition, c'est la mesure qui tranche.

**À savoir dire sans carnet.** On calcule l'AUC de chaque variable seule et on garde `max(auc, 1 − auc)`, parce qu'une variable qui classe à l'envers sépare aussi bien. Le seuil de 0,90 sert d'alarme : une variable aussi forte à elle seule porterait probablement la réponse. Sans le `max`, une variable avec une AUC de 0,02 (parfaite, mais à l'envers) passerait inaperçue.

## Étape 2.1 — Séparer ce qu'on apprend de ce qu'on contrôle

La façon de couper dépend de la nature des données (J5 §1). On vérifie donc deux choses avant de choisir :

- **un client qui revient ?** Il faudrait alors couper par client (`GroupKFold`) ;
- **une date ?** Il faudrait alors couper dans le temps.

In [7]:
from sklearn.model_selection import train_test_split, StratifiedKFold

# un client qui revient ? une colonne de date ?
non_numeriques = list(train.select_dtypes(exclude="number").columns)
constat("aucun client ne revient, aucune date",
        f"{train[ID].nunique()} ID pour {len(train)} lignes ; colonnes non numériques : {non_numeriques}",
        "un client qui revient imposerait GroupKFold, une date une coupe dans le temps",
        "découpage stratifié")

# 80 % pour apprendre, 20 % pour contrôler, avec le même taux de positifs des deux côtés
Xapp, Xctl, yapp, yctl = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)

# 5 plis pour la suite, découpés dans la partie apprentissage seulement
plis = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=0).split(Xapp, yapp))

journal("le découpage", "stratifié en 2 parties (80 % apprentissage / 20 % contrôle), puis 5 plis dans l'apprentissage",
        "aucun client ne revient et aucune date : rien à grouper ni à ordonner ; la cible rare impose la stratification",
        0.00)
print("positifs : apprentissage", round(100 * yapp.mean(), 2), "% · contrôle", round(100 * yctl.mean(), 2), "%")

,observation,chiffre,comparaison,conséquence
0,les tables ont la taille annoncée,"train 160000 × 202, a_decider 40000 × 201",énoncé : 160 000 × 202 et 40 000 × 201,conforme : rien n'est tronqué
1,signe du vide : aucune colonne n'a de trou,0 trou,sur 40360000 cellules (train + a_decider),aucune colonne ne peut trahir la cible par son...
2,signe de la valeur : aucune variable seule ne ...,AUC seule maximale 0.5681 (var_81),"seuil d'alarme 0,90 : 0 variable au-dessus",les 200 variables sont gardées
3,"l'identifiant désigne une ligne, il ne décrit ...",160000 valeurs pour 160000 lignes ; AUC du num...,"0,5 = hasard ; 0 ID en commun avec a_decider",l'écarter
4,"aucun client ne revient, aucune date",160000 ID pour 160000 lignes ; colonnes non nu...,"un client qui revient imposerait GroupKFold, u...",découpage stratifié


,décision,choix,raison (sans le score),effet mesuré,> double de la barre ?
0,ID_code,écarté,il désigne le client au lieu de le décrire : i...,0.0,barre pas encore mesurée
1,le découpage,stratifié en 2 parties (80 % apprentissage / 2...,aucun client ne revient et aucune date : rien ...,0.0,barre pas encore mesurée


positifs : apprentissage 10.05 % · contrôle 10.05 %


| | |
|---|---|
| Découpage choisi | **stratifié**, en **2 parties** : apprentissage 80 % (128 000 lignes), contrôle 20 % (32 000 lignes) |
| Pour la suite | **5 plis** `StratifiedKFold(shuffle=True, random_state=0)`, découpés dans l'apprentissage seulement |
| Raison | 160 000 ID pour 160 000 lignes (aucun client ne revient), et seule `ID_code` n'est pas numérique (aucune date). Rien n'oblige donc à grouper ni à ordonner. La cible est rare (10 %) : la stratification garde le même taux partout. |
| Vérification | positifs : **10,05 %** en apprentissage, **10,05 %** au contrôle |

**À savoir dire sans carnet.** On vérifie d'abord qu'aucun client ne revient et qu'il n'y a pas de date, puis on coupe 80/20 en gardant le même taux de positifs. Le choix vient de la nature des données, pas d'un score. Sans `stratify=y`, le contrôle pourrait avoir un autre taux de positifs, et l'AUC bougerait juste à cause du tirage.

## Étape 2.4 — Vérifier que rien n'a fuité

Règle : **tout ce qui apprend une statistique s'apprend sur `Xapp` seul**, jamais sur `Xctl` ni sur `a_decider`. Le `Pipeline` le garantit : `fit()` n'apprend que sur ce qu'on lui donne.

| Traitement | Appris sur | Ce qui fuirait s'il était appris sur tout |
|---|---|---|
| Remplir un trou | sans objet : 0 trou (1.8) | la moyenne du contrôle, qui servirait à combler |
| Encoder une catégorie | sans objet : seule `ID_code` n'est pas numérique, et elle est écartée | les modalités du contrôle et leur taux de cible |
| Mise à l'échelle (`StandardScaler`) | `Xapp` | la moyenne et l'écart-type du contrôle et de `a_decider` |
| Sélection des variables par AUC seule | `Xapp` et `yapp` | **la cible du contrôle** : on garderait les variables qui marchent justement sur lui |
| Comptage / fréquence des valeurs | `Xapp` | combien de fois chaque valeur apparaît dans le contrôle et dans `a_decider` |

**La démonstration.** On choisit les 20 meilleures variables de deux façons, puis on compare l'AUC sur le contrôle.
**Ce qu'on attend** (écrit avec `prevoir` avant la mesure ; `essai` refuse de mesurer sans) : le choix fait sur toute la table donne un score **plus haut**, parce qu'il a vu la cible du contrôle. Ce gain-là est faux.

In [8]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest
from sklearn.linear_model import LogisticRegression

def auc_par_colonne(X, y):
    """Note de chaque colonne pour SelectKBest : à quel point son AUC seule s'éloigne de 0,5."""
    return np.array([abs(roc_auc_score(y, X[:, j]) - 0.5) for j in range(X.shape[1])])

# MAUVAIS : les 20 variables sont choisies sur toute la table, contrôle compris
choix_partout = X.columns[SelectKBest(auc_par_colonne, k=20).fit(X.values, y).get_support()]
modele = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
prevoir("k=20 choisies sur toute la table", "+", "la sélection a vu la cible du contrôle")
essai("k=20 choisies sur toute la table", changement="SelectKBest sur X entier, avant le découpage",
      mesure=lambda: roc_auc_score(yctl, modele.fit(Xapp[choix_partout], yapp).predict_proba(Xctl[choix_partout])[:, 1]))

# BON : le choix est une étape du pipeline, donc il n'apprend que sur Xapp
chaine = make_pipeline(StandardScaler(), SelectKBest(auc_par_colonne, k=20), LogisticRegression(max_iter=1000))
prevoir("k=20 choisies sur Xapp seul", "−", "plus rien du contrôle n'entre dans le choix")
essai("k=20 choisies sur Xapp seul", changement="SelectKBest dans le pipeline, appris sur Xapp",
      mesure=lambda: roc_auc_score(yctl, chaine.fit(Xapp, yapp).predict_proba(Xctl)[:, 1]))

choix_app = Xapp.columns[chaine[1].get_support()]   # chaine[1] = l'étape SelectKBest
print("variables qui changent :", sorted(set(choix_partout) ^ set(choix_app)))

,essai,attendu,raison,écrit à
0,k=20 choisies sur toute la table,+,la sélection a vu la cible du contrôle,08:56:33


,hypothèse,changement,attendu,mesuré,prédiction écrite avant ?
0,k=20 choisies sur toute la table,"SelectKBest sur X entier, avant le découpage",+,74.65,"oui, à 08:56:33"


,essai,attendu,raison,écrit à
0,k=20 choisies sur toute la table,+,la sélection a vu la cible du contrôle,08:56:33
1,k=20 choisies sur Xapp seul,−,plus rien du contrôle n'entre dans le choix,08:56:34


,hypothèse,changement,attendu,mesuré,prédiction écrite avant ?
0,k=20 choisies sur toute la table,"SelectKBest sur X entier, avant le découpage",+,74.65,"oui, à 08:56:33"
1,k=20 choisies sur Xapp seul,"SelectKBest dans le pipeline, appris sur Xapp",−,74.53,"oui, à 08:56:34"


variables qui changent : ['var_13', 'var_34']


| Choix des 20 variables | AUC contrôle (×100) |
|---|---|
| sur toute la table (fuite) | 74,65 |
| sur `Xapp` seul (dans le pipeline) | 74,53 |
| **écart** | **+0,12** |

Le signe attendu est le bon : la fuite gonfle le score. Elle change 2 variables sur 20 (`var_13` et `var_34`). L'écart est petit, parce que le contrôle ne représente que 20 % de la table. Tant que la barre de bruit (étape 2.2) n'est pas mesurée, on ne peut pas dire s'il est réel. Mais un petit écart faux est plus dangereux qu'un gros : personne ne le remarque.

**À savoir dire sans carnet.** Chaque traitement qui apprend une statistique est mis dans le pipeline, pour qu'il n'apprenne que sur `Xapp`. La démonstration compare la même sélection faite avant et après le découpage. Si on sortait `SelectKBest` du pipeline et qu'on le faisait sur `X` entier, le contrôle participerait au choix, et l'AUC ne mesurerait plus un client nouveau.

## Étape 2.7 — Enregistrer l'essai

Un seul modèle, sans réglage : `StandardScaler` puis `LogisticRegression(max_iter=1000)`, sur les 200 variables.
**Ce qu'on attend** (écrit avec `prevoir`) : il bat le plancher, parce que 200 variables faibles combinées doivent faire mieux qu'un tri sans modèle.

L'AUC est écrite en **points** (×100), comme le plancher : 56,39 et non 0,5639.

In [9]:
import os
from datetime import date

PLANCHER = 56.39  # à remplacer par le plancher mesuré à l'étape 2.3 (AUC en points, donc ×100)
NOM = "régression logistique sans réglage"

modele = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
prevoir(NOM, "+", "200 variables faibles combinées doivent faire mieux qu'un tri sans modèle")
essai(NOM, lambda: roc_auc_score(yctl, modele.fit(Xapp, yapp).predict_proba(Xctl)[:, 1]),
      changement="StandardScaler + LogisticRegression(max_iter=1000), 200 variables")

# on complète la ligne après coup, sans toucher à la fonction essai
ligne = pd.DataFrame(ESSAIS)
ligne = ligne[ligne["hypothèse"] == NOM].copy()
ligne["date"] = date.today().isoformat()
ligne["variables utilisées"] = f"var_0…var_199 ({X.shape[1]})"
ligne["score"] = ligne["mesuré"]
ligne["plancher battu ?"] = np.where(ligne["score"] > PLANCHER, "oui", "non")

# on ajoute la ligne au fichier ; l'en-tête n'est écrit que la première fois
ligne.to_csv("essais.csv", mode="a", header=not os.path.exists("essais.csv"), index=False)
ligne

,essai,attendu,raison,écrit à
0,k=20 choisies sur toute la table,+,la sélection a vu la cible du contrôle,08:56:33
1,k=20 choisies sur Xapp seul,−,plus rien du contrôle n'entre dans le choix,08:56:34
2,régression logistique sans réglage,+,200 variables faibles combinées doivent faire ...,08:56:41


,hypothèse,changement,attendu,mesuré,prédiction écrite avant ?
0,k=20 choisies sur toute la table,"SelectKBest sur X entier, avant le découpage",+,74.65,"oui, à 08:56:33"
1,k=20 choisies sur Xapp seul,"SelectKBest dans le pipeline, appris sur Xapp",−,74.53,"oui, à 08:56:34"
2,régression logistique sans réglage,StandardScaler + LogisticRegression(max_iter=1...,+,86.58,"oui, à 08:56:41"


,hypothèse,changement,attendu,mesuré,prédiction écrite avant ?,date,variables utilisées,score,plancher battu ?
2,régression logistique sans réglage,StandardScaler + LogisticRegression(max_iter=1...,+,86.58,"oui, à 08:56:41",2026-11-13,var_0…var_199 (200),86.58,oui


| Date | Variables | Score (AUC ×100) | Plancher | Plancher battu ? |
|---|---|---|---|---|
| 2026-11-13 | `var_0` … `var_199` (200) | **86,58** | 56,39 | **oui** |

La ligne est ajoutée à `essais.csv`. Relancer la cellule ajoute une nouvelle ligne, sans répéter l'en-tête.

**À savoir dire sans carnet.** On entraîne le modèle sur `Xapp`, on mesure l'AUC sur `Xctl`, puis on écrit la date, les variables, le score et la comparaison au plancher. Le score et le plancher sont dans la même unité (points). Si on écrivait `PLANCHER = 0.5639`, n'importe quel modèle « battrait » le plancher, et la colonne mentirait sans aucune erreur.